# Análise individual: AI Continent Action Plan (União Europeia)

**Projeto:** *As Relações Brasil-China e a Política de Governança Brasileira sobre Inteligência Artificial* (FAPESP, Processo nº 2026/04597-1)
**Etapa:** 02 (análise individual), conforme a skill 02 ([`skill_análise_individual.md`](skill_análise_individual.md))
**Insumo:** `AI_Continent_2025.json`, produzido pela skill 01 ([`skill_extração.md`](skill_extração.md))

---

> ⚠️ **Visualizações prévias.** Todas as visualizações geradas neste notebook (gráficos e tabelas) são **prévias** e compõem **apenas uma base para o início da construção de novos gráficos**. As novas visualizações serão construídas de acordo com os comandos e as exigências do pesquisador, que são obedecidos em última instância. Nenhuma visualização, tabela ou relatório vai para a pasta `resultados/` sem comando expresso do pesquisador.

## 1. Do que se trata

Este notebook analisa, de forma **individual**, o texto do *AI Continent Action Plan*, um dos planos de inteligência artificial da União Europeia. Ele faz parte do fluxo de análise da pasta *Planos Nacionais de IA*: a skill 01 extrai o texto essencial de cada plano; esta etapa (skill 02) analisa cada plano isoladamente; a skill 03 compara os planos entre si.

## 2. Finalidade

A finalidade deste notebook é **gerar visualizações e interpretações sobre o plano analisado**, por meio de gráficos, tabelas e medidas textuais. As análises permitem conhecer:

- a **extensão** e a estrutura do texto;
- os **termos mais comuns** e como se distribuem ao longo do documento;
- as **expressões recorrentes**.

## 3. Insumo e recorte do conteúdo

São utilizados **apenas os resultados da skill 01**, ou seja, o arquivo JSON do plano gerado por aquela etapa. O notebook lê **um único arquivo**, `AI_Continent_2025.json` (`ARQUIVO_JSON`, na configuração). O outro plano da União Europeia na pasta, a *Apply AI Strategy* (`Apply_AI_2025.json`), não entra nesta análise. O relatório de extração (`relatorios_extracao/`) é apenas informativo e **não** é consultado. Do JSON, interessam **somente**:

| Campo | Uso neste notebook |
|---|---|
| `document.title` e `document.subtitle` | Identificação do plano. Por padrão, **entram na contagem** como unidade 0 (`INCLUIR_TITULO_NA_CONTAGEM = True`). Este plano não tem subtítulo |
| `sections[].text` | **Conteúdo analisado**: o texto integral e literal de cada unidade (títulos internos, parágrafos e itens de lista) |
| `sections[].type` | Filtro: os títulos internos (`heading`) entram por padrão (`INCLUIR_TITULOS_INTERNOS = True`) |
| `sections[].category` | Filtro: todas as categorias entram por padrão (`CATEGORIAS_EXCLUIDAS = []`). Neste plano há duas: `Body` (corpo) e `Box` (quadros) |
| `sections[].id` | Apenas índice de posição da unidade no documento |
| `sections[].path` | Apenas agrupador estrutural (por exemplo, por seção), usado somente por comando do pesquisador. Não é usado nas visualizações abaixo |
| `extraction` | Apenas verificação de integridade |

**Todo conteúdo periférico fica fora da análise**: numeração (`number`), nível (`level`), página (`page`) e os demais metadados (`issuer`, `date`, `period` etc.). A função `carregar_extracao` descarta esse conteúdo no momento da leitura. Todo filtro aplicado é informado no perfil do documento (seção 10), com o número de unidades e de tokens que ele retirou.

## 4. Método (visão geral)

| Etapa | Procedimento |
|---|---|
| Verificação | Integridade do JSON: origem na skill 01 e no corpus *Planos Nacionais de IA*, sequência dos identificadores, unidades vazias, número de unidades, título e idioma |
| Recorte | Título como unidade 0; títulos internos e todas as categorias incluídos; nenhuma unidade retirada (parâmetros da configuração) |
| Pré-processamento | spaCy (`en_core_web_sm`): tokenização, reunião de compostos com hífen, lematização (com correções explícitas), classes gramaticais, *stopwords* |
| Exame de inflação | Busca de unidades idênticas, de trechos longos repetidos em unidades diferentes e de resíduos de extração (skill 02, seção 3.5) |
| Frequências | Frequência absoluta, por 1.000 tokens, alcance (unidades em que o termo aparece) e dispersão (DP normalizado de Gries) |
| Expressões | Bigramas de tokens de conteúdo adjacentes no texto original |

O detalhamento de cada procedimento está na configuração, nas funções comentadas e no texto que acompanha cada visualização.

## 5. Transparência das escolhas

**Todo método utilizado e todas as escolhas tomadas pelo pesquisador estão presentes e verificáveis** em dois lugares:

1. **no código**: os parâmetros ficam reunidos na célula de configuração, e cada função traz comentários sobre o que faz;
2. **na descrição que segue cada visualização**, composta por: *Método e escolhas*, *Como ler*, *Interpretação* e *Fragilidades e ajustes possíveis*.

As tabelas completas (e não apenas os recortes exibidos nos gráficos), as figuras e os demais arquivos **só são exportados para a pasta `resultados/` por comando expresso do pesquisador** (skill 02, seção 4.4). Sem esse comando, o notebook apenas exibe os resultados.

## 6. Princípios

Todo o conteúdo deste notebook segue os princípios da **skill 02** (análise individual dos planos). Por isso, **não incorpora análise conjunta**: nenhum resultado aqui é comparado com outro plano. A comparação entre planos é feita em notebook próprio, segundo a skill 03.

> As interpretações foram redigidas **somente após a execução** do notebook, com base nos valores efetivamente obtidos, separando observação de inferência (skill 02, seção 4.3). Se a configuração ou o JSON mudarem, elas precisam ser revistas.

## 7. Configuração

Todos os parâmetros editáveis estão nesta célula. Qualquer alteração deve ser registrada. A gravação em `resultados/` está desativada (`SALVAR_SAIDAS = False`) e só é ativada por comando do pesquisador.

In [ ]:
import difflib
import hashlib
import json
import math
import platform
import re
import unicodedata
import zlib
from datetime import datetime
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import spacy
from IPython.display import Markdown, display
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter, MaxNLocator

pd.set_option("display.max_rows", 300)
pd.set_option("display.max_colwidth", 140)

# =====================================================================
# CONFIGURAÇÃO — parâmetros editáveis pelo pesquisador
# =====================================================================
SIGLA = "AI_Continent_2025"
ARQUIVO_JSON = "AI_Continent_2025.json"         # entregável da skill 01

# Saídas (figuras, tabelas, registro). Resultados anteriores nunca são sobrescritos:
# para uma nova execução com mudanças, altere VERSAO_SAIDA (v2, v3, ...).
NOME_SAIDA = SIGLA
VERSAO_SAIDA = "v1"
# Gravação em resultados/ DESATIVADA: visualizações, tabelas e relatórios só vão para
# resultados/ por comando expresso do pesquisador (skill 02, seção 4.4).
SALVAR_SAIDAS = False
SOBRESCREVER = False

# Recorte do conteúdo (skill 02, seção 3.1)
INCLUIR_TITULO_NA_CONTAGEM = True    # título (e subtítulo, se houver) entram como unidade 0
INCLUIR_TITULOS_INTERNOS = True      # unidades type = "heading" entram na contagem
CATEGORIAS_EXCLUIDAS = []            # ex.: ["Box"]; retirar categorias é decisão do pesquisador
UNIDADES_EXCLUIDAS = {}              # {id: "motivo"}; tratamento de inflação (skill 02, seção 3.5)

# Pré-processamento
IDIOMA_ESPERADO = "en"
MODELO_SPACY = "en_core_web_sm"
CLASSES_DE_CONTEUDO = {"NOUN", "PROPN", "VERB", "ADJ", "ADV"}
TAMANHO_MINIMO_LEMA = 2
STOPWORDS_ADICIONAIS = []    # lemas a remover além da lista do spaCy (decisão do pesquisador)
STOPWORDS_MANTIDAS = []      # lemas da lista do spaCy que devem ser mantidos (decisão do pesquisador)
CORRECOES_DE_LEMA = {
    "datum": "data",         # o spaCy lematiza "data" como "datum"; mantém-se "data", forma usada nos textos
    # Nomes próprios no plural (classe PROPN): o spaCy não os lematiza, e o lema fica igual à
    # forma escrita ("AI Factories" -> "factories", enquanto "AI Factory" -> "factory").
    # Regra única: todo plural regular de nome próprio encontrado no texto recebe o lema
    # singular, como o spaCy já faz com os substantivos comuns (skill 02, seção 3.3, passos 4 e 5).
    # Lista levantada no exame do texto processado deste plano. Para manter o mesmo
    # pré-processamento em todos os planos (skill 03), a lista deve ser a mesma em todos os
    # notebooks da pasta. Para reverter, basta apagar as entradas.
    "algorithms": "algorithm",
    "antennas": "antenna",
    "centres": "centre",
    "champions": "champion",
    "chips": "chip",
    "competitions": "competition",
    "europeans": "european",
    "facilities": "facility",
    "factories": "factory",
    "gigafactories": "gigafactory",
    "guidelines": "guideline",
    "hubs": "hub",
    "labs": "lab",
    "offices": "office",
    "partnerships": "partnership",
    "phds": "phd",
    "projects": "project",
    "researchers": "researcher",
    "skills": "skill",
    "spaces": "space",
    "states": "state",
    "stories": "story",
    "strategies": "strategy",
    "students": "student",
    "technologies": "technology",
    "undertakings": "undertaking",
    "vessels": "vessel",
}

# Exame de inflação (skill 02, seção 3.5): tamanho mínimo, em palavras, de um trecho
# idêntico repetido em unidades diferentes para que ele seja apontado no diagnóstico.
MIN_PALAVRAS_TRECHO_REPETIDO = 12

# Recortes de visualização (as tabelas exportadas são sempre completas)
TOP_N_LEMAS = 25
TOP_N_DISPERSAO = 15
TOP_N_BIGRAMAS = 20
FREQ_MINIMA_BIGRAMA = 2

# Incerteza (bootstrap por unidades)
N_BOOTSTRAP = 2000
NIVEL_CONFIANCA = 0.95
SEMENTE = 2026

## 8. Ambiente, estilo e funções

Restrição de pasta, pasta de saída (usada apenas quando o pesquisador ordenar a gravação em `resultados/`), padrão gráfico e funções auxiliares.

In [ ]:
# ---------------------------------------------------------------------------
# Restrição de ambiente: o notebook só roda de dentro da pasta "Planos Nacionais de IA".
# ---------------------------------------------------------------------------
PASTA = Path.cwd()
if unicodedata.normalize("NFC", PASTA.name) != "Planos Nacionais de IA":
    raise RuntimeError(
        "Este notebook deve ser executado de dentro da pasta 'Planos Nacionais de IA' "
        f"(restrição de ambiente da skill). Pasta atual: {PASTA}"
    )

# ---------------------------------------------------------------------------
# Pasta de saída versionada: resultados anteriores nunca são sobrescritos.
# ---------------------------------------------------------------------------
DIR_SAIDA = PASTA / "resultados" / NOME_SAIDA / VERSAO_SAIDA
if SALVAR_SAIDAS:
    if DIR_SAIDA.exists() and any(DIR_SAIDA.iterdir()) and not SOBRESCREVER:
        raise FileExistsError(
            f"A pasta de saída '{DIR_SAIDA.relative_to(PASTA)}' já contém arquivos. "
            "Para preservar a versão anterior, altere VERSAO_SAIDA (ex.: 'v2') na configuração. "
            "Use SOBRESCREVER = True somente se a substituição for intencional."
        )
    DIR_SAIDA.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# Estilo das figuras (padrão acadêmico: fonte >= 9 pt, 300 dpi, PNG + PDF)
# ---------------------------------------------------------------------------
# Cor fixa por plano, igual em todos os notebooks da pasta (paleta inicial da
# skill 02, seção 3.7, editável pelo pesquisador). Planos do mesmo país ou bloco
# recebem tons da mesma cor. Verde e vermelho se confundem para leitores com
# daltonismo: gráficos com mais de um plano identificam cada um também por
# rótulo direto ou marcador.
COR_DOCUMENTO = {
    "PBIA_2025": "#2E9E4F",    # Brasil (verde)
    "EUA_2025": "#1F3F8F",     # Estados Unidos (azul-escuro)
    "AI_Continent_2025": "#EDB120",   # União Europeia (amarelo)
    "China_2025": "#B2182B",   # China (vermelho)
}
COR_TINTA = "#0b0b0b"     # texto principal
COR_TINTA_2 = "#52514e"   # texto secundário (rótulos, anotações)
COR_GRADE = "#e4e3df"     # grade e linhas de apoio
COR_LIGACAO = "#b9b8b2"   # segmentos de ligação (gráficos de halteres)

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": COR_TINTA_2,
    "axes.linewidth": 0.6,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": COR_GRADE,
    "grid.linewidth": 0.6,
    "grid.linestyle": "-",
    "text.color": COR_TINTA,
    "axes.labelcolor": COR_TINTA,
    "xtick.color": COR_TINTA_2,
    "ytick.color": COR_TINTA_2,
    "legend.frameon": False,
    "pdf.fonttype": 42,
})

# ---------------------------------------------------------------------------
# Funções auxiliares de formatação e gravação
# ---------------------------------------------------------------------------
def num(x, casas=2):
    """Formata número no padrão brasileiro (vírgula decimal, ponto de milhar)."""
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return "—"
    texto = f"{x:,.{casas}f}"
    return texto.replace(",", "§").replace(".", ",").replace("§", ".")


# Rótulos de eixo com vírgula decimal.
FMT_BR = FuncFormatter(lambda v, _: f"{v:g}".replace(".", ","))


def exibir(df, casas=2):
    """Exibe uma tabela com vírgula decimal, sem alterar os valores."""
    display(df.style.format(precision=casas, decimal=",", thousands=".", na_rep="—"))


def salvar_figura(fig, nome):
    """Grava a figura em PNG (300 dpi) e PDF na pasta de saída versionada."""
    if SALVAR_SAIDAS:
        for ext in ("png", "pdf"):
            fig.savefig(DIR_SAIDA / f"{nome}.{ext}")


def salvar_tabela(df, nome, index=True):
    """Grava a tabela completa em CSV (UTF-8 com BOM, ';', vírgula decimal)."""
    if SALVAR_SAIDAS:
        df.to_csv(DIR_SAIDA / f"{nome}.csv", sep=";", decimal=",", encoding="utf-8-sig", index=index)


def semente_documento(sigla):
    """Semente do bootstrap própria de cada plano, derivada de SEMENTE e da sigla.
    Garante (a) os mesmos intervalos no notebook individual e no comparativo e
    (b) sequências aleatórias independentes entre planos diferentes."""
    return np.random.SeedSequence([SEMENTE, zlib.crc32(sigla.encode("utf-8"))])


print(f"Pasta de trabalho: {PASTA}")
print(f"Saídas: {DIR_SAIDA.relative_to(PASTA) if SALVAR_SAIDAS else 'gravação desativada'}")

if SIGLA not in COR_DOCUMENTO:
    raise KeyError(f"O plano '{SIGLA}' não tem cor definida em COR_DOCUMENTO (skill 02, seção 3.7).")
COR = COR_DOCUMENTO[SIGLA]

In [ ]:
# ===========================================================================
# PIPELINE DE PRÉ-PROCESSAMENTO E MEDIDAS
# (idêntico em todos os notebooks de análise da pasta "Planos Nacionais de IA")
# ===========================================================================

# Token (palavra) = sequência de letras, admitindo hífen interno (multi-stakeholder)
# ou sigla com pontos (U.S.). Números, pontuação e símbolos NÃO são tokens.
RE_PALAVRA = re.compile(r"(?:[^\W\d_]\.){2,}|[^\W\d_]+(?:[-‐‑][^\W\d_]+)*")
HIFENS = {"-", "‐", "‑"}

# Corpus desta pasta: o JSON precisa ter sido produzido pela skill 01 para ele.
CORPUS = "Planos Nacionais de IA"

# Tipos de unidade, na ordem em que aparecem no perfil do documento.
ROTULO_TIPO = {
    "title": "título e subtítulo (unidade 0)",
    "heading": "títulos internos",
    "paragraph": "parágrafos",
    "list_item": "itens de lista",
}


def normalizar_hifen(texto):
    """Uniformiza variantes tipográficas do hífen (apenas nas chaves de contagem;
    o texto do JSON não é alterado)."""
    return texto.replace("‐", "-").replace("‑", "-")


def carregar_extracao(caminho):
    """Lê o JSON produzido pela skill 01 e devolve SOMENTE o título, o subtítulo e as unidades
    (id, texto integral, tipo, categoria e path). Todo o conteúdo periférico (number, level,
    page, issuer, date etc.) é descartado aqui. Interrompe a execução se o arquivo não passar
    nas verificações de integridade (skill 02, seção 4.2, item 4)."""
    if not caminho.exists():
        raise FileNotFoundError(
            f"Arquivo '{caminho.name}' não encontrado na pasta. "
            "Execute antes a skill 01 (skill_extração.md) para gerar o JSON do plano."
        )
    bruto = caminho.read_bytes()
    dados = json.loads(bruto.decode("utf-8"))

    problemas = [f"chave '{c}' ausente" for c in ("document", "extraction", "sections") if c not in dados]
    if problemas:
        raise ValueError("Falha na verificação do JSON:\n- " + "\n- ".join(problemas))

    extracao = dados["extraction"]
    if extracao.get("skill") != "skill_extração":
        problemas.append("o JSON não registra ter sido produzido pela skill 01 (extraction.skill)")
    if extracao.get("corpus") != CORPUS:
        problemas.append(
            f"o JSON não registra ter sido produzido para o corpus '{CORPUS}' "
            f"(extraction.corpus = {extracao.get('corpus')!r})"
        )
    secoes = dados["sections"]
    if [s.get("id") for s in secoes] != list(range(1, len(secoes) + 1)):
        problemas.append("os ids das unidades não são sequenciais a partir de 1")
    vazias = [s.get("id") for s in secoes if not isinstance(s.get("text"), str) or not s["text"].strip()]
    if vazias:
        problemas.append(f"unidades com texto vazio: {vazias}")
    if extracao.get("units_extracted") is not None and extracao["units_extracted"] != len(secoes):
        problemas.append(
            f"extraction.units_extracted = {extracao['units_extracted']}, mas há {len(secoes)} unidades"
        )
    if not dados["document"].get("title"):
        problemas.append("título (document.title) ausente")
    if dados["document"].get("language") != IDIOMA_ESPERADO:
        problemas.append(
            f"idioma declarado no JSON: {dados['document'].get('language')!r}; esperado: {IDIOMA_ESPERADO!r}"
        )
    if problemas:
        raise ValueError("Falha na verificação do JSON:\n- " + "\n- ".join(problemas))

    return {
        "arquivo": caminho.name,
        "sha256": hashlib.sha256(bruto).hexdigest(),
        "titulo": dados["document"]["title"],
        "subtitulo": dados["document"].get("subtitle"),
        "idioma": dados["document"].get("language"),
        # SOMENTE id (posição), texto (conteúdo), tipo e categoria (filtros) e path (agrupador)
        "unidades": [
            {"id": s["id"], "texto": s["text"], "tipo": s.get("type"),
             "categoria": s.get("category"), "path": tuple(s.get("path") or ())}
            for s in secoes
        ],
    }


def selecionar_unidades(documento):
    """Aplica o recorte da configuração (skill 02, seção 3.1) e devolve
    (a) as unidades analisadas, na ordem do documento, e
    (b) as unidades retiradas, cada uma com o filtro responsável.
    O título (e o subtítulo, se houver) forma a unidade 0. Uma unidade alcançada por mais de
    um filtro é atribuída ao primeiro, na ordem: título, títulos internos, categorias, ids."""
    ids_json = {u["id"] for u in documento["unidades"]}
    inexistentes = sorted(set(UNIDADES_EXCLUIDAS) - ids_json)
    if inexistentes:
        raise ValueError(f"UNIDADES_EXCLUIDAS cita ids que não existem no JSON: {inexistentes}")
    titulo = documento["titulo"] + (f"\n{documento['subtitulo']}" if documento["subtitulo"] else "")
    candidatas = [{"id": 0, "texto": titulo, "tipo": "title", "categoria": None, "path": ()}]
    candidatas += documento["unidades"]
    analisadas, retiradas = [], []
    for u in candidatas:
        if u["tipo"] == "title" and not INCLUIR_TITULO_NA_CONTAGEM:
            filtro = "INCLUIR_TITULO_NA_CONTAGEM"
        elif u["tipo"] == "heading" and not INCLUIR_TITULOS_INTERNOS:
            filtro = "INCLUIR_TITULOS_INTERNOS"
        elif u["categoria"] in CATEGORIAS_EXCLUIDAS:
            filtro = "CATEGORIAS_EXCLUIDAS"
        elif u["id"] in UNIDADES_EXCLUIDAS:
            filtro = "UNIDADES_EXCLUIDAS"
        else:
            analisadas.append(u)
            continue
        retiradas.append({**u, "filtro": filtro})
    return analisadas, retiradas


def montar_stopwords(nlp):
    """Lista de stopwords do spaCy, ajustada pelas listas editáveis da configuração."""
    base = {w.lower() for w in nlp.Defaults.stop_words}
    return (base | {w.lower() for w in STOPWORDS_ADICIONAIS}) - {w.lower() for w in STOPWORDS_MANTIDAS}


def unir_compostos_hifenizados(doc):
    """Reúne em um único token as palavras compostas com hífen (ex.: multi-stakeholder,
    human-centric, state-of-the-art), que o spaCy separa em vários tokens.
    Só une quando não há espaço em volta do hífen."""
    trechos, i, n = [], 0, len(doc)
    while i < n:
        j = i
        while (j + 2 < n and doc[j].is_alpha and doc[j].whitespace_ == ""
               and doc[j + 1].text in HIFENS and doc[j + 1].whitespace_ == ""
               and doc[j + 2].is_alpha):
            j += 2
        if j > i:
            trechos.append(doc[i:j + 1])
            i = j + 1
        else:
            i += 1
    with doc.retokenize() as rt:
        for trecho in trechos:
            ultimo = trecho[-1]
            # Lema do composto = forma escrita; se o último elemento estiver no plural,
            # usa-se o lema dele (multi-stakeholders -> multi-stakeholder).
            if ultimo.tag_ in {"NNS", "NNPS"}:
                lema = trecho.text[: len(trecho.text) - len(ultimo.text)] + ultimo.lemma_
            else:
                lema = trecho.text
            rt.merge(trecho, attrs={"LEMMA": lema, "POS": ultimo.pos_, "TAG": ultimo.tag_})
    return doc


def processar_documento(unidades, nlp, stopwords):
    """Aplica o pré-processamento a cada unidade (título, título interno, parágrafo ou item)
    e devolve uma tabela com uma linha por token (palavra), na ordem original do documento."""
    linhas, posicao = [], 0
    for (uid, _), doc in zip(unidades, nlp.pipe([texto for _, texto in unidades])):
        doc = unir_compostos_hifenizados(doc)
        for k, frase in enumerate(doc.sents):
            for tok in frase:
                if not RE_PALAVRA.fullmatch(tok.text):
                    continue
                lema_spacy = normalizar_hifen(tok.lemma_.lower())
                linhas.append({
                    "unidade": uid,
                    "frase": f"{uid}.{k}",
                    "i": tok.i,                      # índice na unidade (detecta adjacência)
                    "posicao": posicao,              # posição no documento inteiro
                    "forma": normalizar_hifen(tok.text.lower()),
                    "lema_spacy": lema_spacy,        # lema dado pelo spaCy (antes das correções)
                    "lema": CORRECOES_DE_LEMA.get(lema_spacy, lema_spacy),
                    "classe": tok.pos_,
                    "etiqueta": tok.tag_,
                })
                posicao += 1
    tokens = pd.DataFrame(linhas)
    tokens["conteudo"] = (
        tokens["classe"].isin(CLASSES_DE_CONTEUDO)
        & ~tokens["lema"].isin(stopwords)
        & (tokens["lema"].str.len() >= TAMANHO_MINIMO_LEMA)
        & tokens["lema"].map(lambda lema: bool(RE_PALAVRA.fullmatch(lema)))
    )
    return tokens


def tabela_filtros(retiradas, nlp, stopwords):
    """Quantas unidades e quantos tokens cada filtro do recorte retirou (skill 02, seção 3.1)."""
    valores = {
        "INCLUIR_TITULO_NA_CONTAGEM": INCLUIR_TITULO_NA_CONTAGEM,
        "INCLUIR_TITULOS_INTERNOS": INCLUIR_TITULOS_INTERNOS,
        "CATEGORIAS_EXCLUIDAS": CATEGORIAS_EXCLUIDAS,
        "UNIDADES_EXCLUIDAS": UNIDADES_EXCLUIDAS,
    }
    tam = pd.Series(dtype=int)
    if retiradas:
        tam = processar_documento([(u["id"], u["texto"]) for u in retiradas], nlp, stopwords).groupby("unidade").size()
    linhas = []
    for filtro, valor in valores.items():
        ids = [u["id"] for u in retiradas if u["filtro"] == filtro]
        linhas.append({
            "filtro": filtro,
            "valor": repr(valor),
            "unidades_retiradas": len(ids),
            "tokens_retirados": int(tam.reindex(ids, fill_value=0).sum()),
            "ids_retirados": ", ".join(map(str, ids)) if ids else "—",
        })
    return pd.DataFrame(linhas).set_index("filtro")


def perfil_documento(tokens, tipo_unidade, rotulo):
    """Indicadores descritivos de extensão e vocabulário.
    tipo_unidade: {id: tipo} das unidades analisadas (para contá-las por tipo)."""
    conteudo = tokens[tokens["conteudo"]]
    tam = tokens.groupby("unidade").size()
    freq_cont = conteudo["lema"].value_counts()
    por_tipo = pd.Series(tipo_unidade).value_counts()
    return pd.Series({
        "Unidades analisadas": len(tipo_unidade),
        **{f"Unidades: {nome}": int(por_tipo.get(tipo, 0)) for tipo, nome in ROTULO_TIPO.items()},
        "Unidades sem nenhum token": len(tipo_unidade) - tokens["unidade"].nunique(),
        "Tokens (palavras)": len(tokens),
        "Formas distintas": tokens["forma"].nunique(),
        "Tokens de conteúdo": len(conteudo),
        "Tokens de conteúdo (% dos tokens)": len(conteudo) / len(tokens) * 100,
        "Lemas de conteúdo distintos": conteudo["lema"].nunique(),
        "Lemas de conteúdo com uma única ocorrência": int((freq_cont == 1).sum()),
        "Tokens por unidade: média": tam.mean(),
        "Tokens por unidade: mediana": tam.median(),
        "Tokens por unidade: mínimo": tam.min(),
        "Tokens por unidade: máximo": tam.max(),
    }, name=rotulo)


def tabela_correcoes_lema(tokens):
    """Ocorrências afetadas por cada correção explícita de lema (CORRECOES_DE_LEMA)."""
    corr = tokens[tokens["lema_spacy"] != tokens["lema"]]
    tab = corr.groupby(["lema_spacy", "lema"]).agg(
        ocorrencias=("forma", "size"),
        formas_observadas=("forma", lambda s: "; ".join(f"{f} ({n})" for f, n in s.value_counts().items())),
    ).reset_index().rename(columns={"lema_spacy": "lema_do_spacy", "lema": "lema_corrigido"})
    tab = tab.sort_values(["ocorrencias", "lema_do_spacy"], ascending=[False, True])
    return tab.set_index("lema_do_spacy")


def lemas_descartados_por_stopwords(tokens, stopwords):
    """Lemas de classes de conteúdo (CLASSES_DE_CONTEUDO) que a lista de stopwords retirou
    da contagem de conteúdo, com as suas frequências, classes e formas observadas."""
    sel = tokens[tokens["classe"].isin(CLASSES_DE_CONTEUDO) & tokens["lema"].isin(stopwords)]
    grupos = sel.groupby("lema")
    tab = pd.DataFrame({
        "frequencia": grupos.size(),
        "por_1000_tokens": grupos.size() / len(tokens) * 1000,
        "classes": grupos["classe"].agg(lambda s: "; ".join(f"{c} ({n})" for c, n in s.value_counts().items())),
        "formas_observadas": grupos["forma"].agg(
            lambda s: "; ".join(f"{f} ({n})" for f, n in s.value_counts().items())
        ),
    })
    tab.index.name = "lema"
    tab = tab.reset_index().sort_values(["frequencia", "lema"], ascending=[False, True])
    return tab.set_index("lema")


def examinar_inflacao(unidades, n_palavras=12):
    """Exame de inflação (skill 02, seção 3.5). Apenas DIAGNOSTICA, sem retirar nada:
    (a) unidades com texto idêntico (após minúsculas e espaços uniformizados);
    (b) trechos de n_palavras ou mais palavras idênticas, repetidos em unidades diferentes.
    Se o diagnóstico revelar artefato (texto multiplicado sem que o plano o tenha escrito
    mais vezes), o tratamento é feito em UNIDADES_EXCLUIDAS, com o motivo."""
    grupos = {}
    for uid, texto in unidades:
        grupos.setdefault(" ".join(normalizar_hifen(texto.lower()).split()), []).append(uid)
    identicas = pd.DataFrame(
        [{"texto": t, "unidades": ", ".join(map(str, ids)), "n_unidades": len(ids)}
         for t, ids in grupos.items() if len(ids) > 1],
        columns=["texto", "unidades", "n_unidades"])

    palavras = {uid: [p.lower() for p in RE_PALAVRA.findall(normalizar_hifen(texto))] for uid, texto in unidades}
    indice = {}
    for uid, ps in palavras.items():
        for k in range(len(ps) - n_palavras + 1):
            indice.setdefault(tuple(ps[k:k + n_palavras]), set()).add(uid)
    pares = set()
    for ids in indice.values():
        ids = sorted(ids)
        pares.update((a, b) for i, a in enumerate(ids) for b in ids[i + 1:])
    linhas = []
    for a, b in sorted(pares):
        blocos = difflib.SequenceMatcher(None, palavras[a], palavras[b], autojunk=False).get_matching_blocks()
        for bloco in blocos:
            if bloco.size >= n_palavras:
                linhas.append({"unidade_a": a, "unidade_b": b, "palavras": bloco.size,
                               "trecho": " ".join(palavras[a][bloco.a:bloco.a + bloco.size])})
    repetidos = pd.DataFrame(linhas, columns=["unidade_a", "unidade_b", "palavras", "trecho"])
    return identicas, repetidos


def formas_alfanumericas(unidades, nlp):
    """Tokens do spaCy que misturam letras e dígitos (ex.: siglas como GenAI4EU, trimestres
    como Q2, ou marcadores de nota colados a palavras, que seriam resíduo de extração).
    Pela definição de token (só letras), eles não entram em nenhuma contagem."""
    formas = [t.text for _, texto in unidades for t in nlp.tokenizer(texto)
              if any(c.isalpha() for c in t.text) and any(c.isdigit() for c in t.text)]
    return pd.Series(formas, dtype=object).value_counts()


def tabela_frequencias(tokens):
    """Frequência, taxa por 1.000 tokens, alcance e dispersão (DP normalizado) de
    TODOS os lemas de conteúdo. Ordenação: frequência decrescente; empate -> ordem alfabética."""
    n_tokens = len(tokens)
    unidades = sorted(tokens["unidade"].unique())
    proporcao_unidade = (tokens.groupby("unidade").size().reindex(unidades) / n_tokens).to_numpy()
    conteudo = tokens[tokens["conteudo"]]
    cruz = pd.crosstab(conteudo["lema"], conteudo["unidade"]).reindex(columns=unidades, fill_value=0)
    freq = cruz.sum(axis=1)
    # DP de Gries (2008), normalizado por Lijffijt & Gries (2012):
    # 0 = distribuição proporcional ao tamanho das unidades; 1 = concentração máxima.
    v = cruz.div(freq, axis=0).to_numpy()
    dp = 0.5 * np.abs(v - proporcao_unidade).sum(axis=1)
    dp_norm = dp / (1 - proporcao_unidade.min()) if len(unidades) > 1 else np.full(len(freq), np.nan)
    formas = conteudo.groupby("lema")["forma"].agg(
        lambda s: "; ".join(f"{forma} ({n})" for forma, n in s.value_counts().items())
    )
    tab = pd.DataFrame({
        "frequencia": freq,
        "por_1000_tokens": freq / n_tokens * 1000,
        "unidades_com_ocorrencia": (cruz > 0).sum(axis=1),
        "pct_unidades": (cruz > 0).sum(axis=1) / len(unidades) * 100,
        "DP_norm": dp_norm,
        "formas_observadas": formas,
    })
    tab.index.name = "lema"
    tab = tab.reset_index().sort_values(["frequencia", "lema"], ascending=[False, True])
    return tab.set_index("lema")


def relatar_empates(tab, coluna, n, rotulo):
    """Informa termos empatados no ponto de corte de um recorte 'top N'."""
    if len(tab) <= n:
        return
    corte = tab[coluna].iloc[n - 1]
    fora = tab.iloc[n:]
    fora = fora[fora[coluna] == corte]
    if len(fora):
        lista = ", ".join(fora.index[:20]) + (" …" if len(fora) > 20 else "")
        print(
            f"Empate no ponto de corte: o {n}º {rotulo} tem valor {num(corte, 2)}; "
            f"outros {len(fora)} com o mesmo valor ficaram fora do recorte "
            f"(desempate por ordem alfabética): {lista}"
        )


def tabela_bigramas(tokens):
    """Pares de tokens de conteúdo ADJACENTES no texto original (mesma frase, sem
    pontuação entre eles). Não se formam pares artificiais pela remoção de stopwords."""
    a = tokens.iloc[:-1].reset_index(drop=True)
    b = tokens.iloc[1:].reset_index(drop=True)
    ok = (
        (a["frase"].to_numpy() == b["frase"].to_numpy())
        & (b["i"].to_numpy() - a["i"].to_numpy() == 1)
        & a["conteudo"].to_numpy() & b["conteudo"].to_numpy()
    )
    pares = pd.DataFrame({
        "bigrama": (a["lema"] + " " + b["lema"])[ok],
        "forma": (a["forma"] + " " + b["forma"])[ok],
        "unidade": a["unidade"][ok],
    })
    if pares.empty:
        return pd.DataFrame(columns=["frequencia", "por_1000_tokens", "unidades_com_ocorrencia",
                                     "forma_principal", "formas_observadas"]).rename_axis("bigrama")
    grupos = pares.groupby("bigrama")
    tab = pd.DataFrame({
        "frequencia": grupos.size(),
        "por_1000_tokens": grupos.size() / len(tokens) * 1000,
        "unidades_com_ocorrencia": grupos["unidade"].nunique(),
        "forma_principal": grupos["forma"].agg(lambda s: s.value_counts().index[0]),
        "formas_observadas": grupos["forma"].agg(
            lambda s: "; ".join(f"{f} ({n})" for f, n in s.value_counts().items())
        ),
    })
    tab = tab.reset_index().sort_values(["frequencia", "bigrama"], ascending=[False, True])
    return tab.set_index("bigrama")


# ---------------------------------------------------------------------------
# Funções reservadas para análises feitas por comando do pesquisador (categorias
# temáticas com codebook e intervalo de confiança, verbos modais). Não são usadas
# nas visualizações-padrão abaixo.
# ---------------------------------------------------------------------------
def carregar_codebook(caminho):
    """Lê o codebook e verifica a exclusividade mútua das categorias.
    Devolve o codebook e a lista de entradas ordenada da expressão mais longa para a mais curta."""
    codebook = json.loads(caminho.read_text(encoding="utf-8"))
    entradas, vistos = [], {}
    for categoria, info in codebook["categorias"].items():
        for termo in info["termos"]:
            chave = tuple(normalizar_hifen(termo.lower()).split())
            if chave in vistos:
                raise ValueError(
                    f"Termo '{termo}' aparece em '{vistos[chave]}' e em '{categoria}'. "
                    "As categorias do codebook devem ser mutuamente exclusivas."
                )
            vistos[chave] = categoria
            entradas.append((chave, categoria, termo))
    entradas.sort(key=lambda e: -len(e[0]))
    sha = hashlib.sha256(caminho.read_bytes()).hexdigest()
    return codebook, entradas, sha


def aplicar_codebook(tokens, entradas):
    """Localiza os termos do codebook no texto. Cada palavra do termo é comparada com a
    forma OU com o lema do token. Expressões exigem palavras contíguas na mesma frase.
    Prevalece a expressão mais longa, e cada token é contado no máximo uma vez."""
    achados = []
    for _, frase in tokens.groupby("frase", sort=False):
        formas = frase["forma"].tolist()
        lemas = frase["lema"].tolist()
        idx = frase["i"].tolist()
        pos = frase["posicao"].tolist()
        uid = frase["unidade"].iloc[0]
        j, n = 0, len(formas)
        while j < n:
            for chave, categoria, termo in entradas:
                tam = len(chave)
                if j + tam > n:
                    continue
                if all(formas[j + k] == chave[k] or lemas[j + k] == chave[k] for k in range(tam)) and \
                   all(idx[j + k + 1] - idx[j + k] == 1 for k in range(tam - 1)):
                    achados.append({"unidade": uid, "posicao": pos[j], "categoria": categoria,
                                    "termo": termo, "forma": " ".join(formas[j:j + tam]), "n_tokens": tam})
                    j += tam
                    break
            else:
                j += 1
    return pd.DataFrame(achados, columns=["unidade", "posicao", "categoria", "termo", "forma", "n_tokens"])


def taxas_bootstrap(contagens, tamanhos, semente):
    """Bootstrap por unidades: reamostra unidades com reposição e recalcula a taxa por
    1.000 tokens. Respeita a dependência entre palavras de uma mesma unidade.
    contagens: matriz (rótulos x unidades). Devolve (rótulos x reamostras)."""
    rng = np.random.default_rng(semente)
    n_u = len(tamanhos)
    idx = rng.integers(0, n_u, size=(N_BOOTSTRAP, n_u))
    return contagens[:, idx].sum(axis=2) / tamanhos[idx].sum(axis=1) * 1000


def tabela_categorias(achados, tokens, categorias, semente):
    """Ocorrências por categoria, taxa por 1.000 tokens com IC por bootstrap,
    participação no total do codebook e alcance."""
    unidades = sorted(tokens["unidade"].unique())
    tamanhos = tokens.groupby("unidade").size().reindex(unidades).to_numpy()
    if achados.empty:
        matriz = pd.DataFrame(0, index=categorias, columns=unidades)
    else:
        matriz = pd.crosstab(achados["categoria"], achados["unidade"]).reindex(
            index=categorias, columns=unidades, fill_value=0)
    reamostras = taxas_bootstrap(matriz.to_numpy(), tamanhos, semente)
    alfa = (1 - NIVEL_CONFIANCA) / 2
    ocorr = matriz.sum(axis=1)
    tab = pd.DataFrame({
        "ocorrencias": ocorr,
        "por_1000_tokens": ocorr / len(tokens) * 1000,
        "ic_inferior": np.quantile(reamostras, alfa, axis=1),
        "ic_superior": np.quantile(reamostras, 1 - alfa, axis=1),
        "pct_do_codebook": ocorr / ocorr.sum() * 100 if ocorr.sum() else np.nan,
        "unidades_com_ocorrencia": (matriz > 0).sum(axis=1),
    })
    tab.index.name = "categoria"
    return tab, reamostras


def tabela_termos_codebook(achados, entradas, n_tokens, categorias):
    """Contribuição de cada termo do codebook, incluindo os termos sem ocorrência."""
    linhas = []
    for _, categoria, termo in sorted(entradas, key=lambda e: (list(categorias).index(e[1]), e[2])):
        sub = achados[(achados["categoria"] == categoria) & (achados["termo"] == termo)]
        linhas.append({
            "categoria": categoria,
            "termo": termo,
            "ocorrencias": len(sub),
            "por_1000_tokens": len(sub) / n_tokens * 1000,
            "formas_observadas": "; ".join(f"{f} ({n})" for f, n in sub["forma"].value_counts().items()),
        })
    tab = pd.DataFrame(linhas)
    total_cat = tab.groupby("categoria")["ocorrencias"].transform("sum")
    tab["pct_da_categoria"] = np.where(total_cat > 0, tab["ocorrencias"] / total_cat.where(total_cat > 0, 1) * 100, np.nan)
    return tab


def robustez_termo_dominante(termos, n_tokens, categorias):
    """Teste de robustez: quanto cada categoria depende do seu termo mais frequente."""
    linhas = []
    for categoria in categorias:
        sub = termos[termos["categoria"] == categoria].sort_values(
            ["ocorrencias", "termo"], ascending=[False, True])
        total = sub["ocorrencias"].sum()
        dominante = sub.iloc[0]
        linhas.append({
            "categoria": categoria,
            "termo_dominante": dominante["termo"] if total else "—",
            "ocorrencias_do_termo": dominante["ocorrencias"] if total else 0,
            "participacao_do_termo_pct": dominante["ocorrencias"] / total * 100 if total else np.nan,
            "por_1000_com_o_termo": total / n_tokens * 1000,
            "por_1000_sem_o_termo": (total - dominante["ocorrencias"]) / n_tokens * 1000,
        })
    tab = pd.DataFrame(linhas).set_index("categoria")
    tab["alerta"] = np.where(tab["participacao_do_termo_pct"] > 50,
                             "mais da metade da categoria vem de um único termo", "")
    return tab


MODAIS = ["must", "shall", "should", "will", "would", "may", "might", "can", "could"]


def tabela_modais(tokens):
    """Verbos modais (etiqueta gramatical MD), por lema, por 1.000 tokens.
    Ordem fixa; modais fora da lista padrão, se houver, entram ao final."""
    modais = tokens[tokens["etiqueta"] == "MD"]["lema"].value_counts()
    ordem = MODAIS + sorted(set(modais.index) - set(MODAIS))
    modais = modais.reindex(ordem, fill_value=0)
    return pd.DataFrame({"ocorrencias": modais, "por_1000_tokens": modais / len(tokens) * 1000}).rename_axis("modal")


def grafico_dispersao_posicional(ax, posicoes_por_linha, rotulos, inicios_unidades, cor):
    """Gráfico de dispersão posicional: cada traço é uma ocorrência, na posição
    relativa em que aparece no documento (0 = início; 1 = fim)."""
    if len(inicios_unidades) <= 120:
        for x in inicios_unidades:
            ax.axvline(x, color=COR_GRADE, lw=0.5, zorder=0)
    for y, posicoes in enumerate(posicoes_por_linha):
        ax.vlines(posicoes, y - 0.36, y + 0.36, color=cor, lw=0.9)
    ax.set_yticks(range(len(rotulos)), labels=rotulos)
    ax.set_ylim(len(rotulos) - 0.5, -0.5)
    ax.set_xlim(0, 1)
    ax.grid(False)
    ax.xaxis.set_major_formatter(FMT_BR)
    ax.set_xlabel("Posição relativa no documento (0 = início; 1 = fim)")


def versoes_ambiente(nlp):
    return {
        "python": platform.python_version(),
        "spacy": spacy.__version__,
        "modelo_spacy": f"{MODELO_SPACY} {nlp.meta.get('version', '?')}",
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "matplotlib": matplotlib.__version__,
    }

## 9. Carregamento e verificação do insumo

O JSON da skill 01 é lido em modo somente leitura. A execução é interrompida se: o arquivo não existir; não tiver sido produzido pela skill 01 para o corpus *Planos Nacionais de IA*; os identificadores não forem sequenciais; houver unidade vazia; o número de unidades divergir do registrado na extração; faltar o título; ou o idioma não for o esperado (`en`). O SHA-256 do JSON é exibido para registro. Em seguida, aplica-se o recorte da configuração (título como unidade 0, títulos internos e categorias).

In [ ]:
documento = carregar_extracao(PASTA / ARQUIVO_JSON)
analisadas, retiradas = selecionar_unidades(documento)
unidades = [(u["id"], u["texto"]) for u in analisadas]      # (id, texto): o que é analisado
TIPO_UNIDADE = {u["id"]: u["tipo"] for u in analisadas}      # usados só para descrever o perfil
CATEGORIA_UNIDADE = {u["id"]: u["categoria"] or "título" for u in analisadas}

print(f"Título: {documento['titulo']}")
print(f"Subtítulo: {documento['subtitulo'] or '— (o plano não tem subtítulo)'}")
print(f"Arquivo: {documento['arquivo']}")
print(f"SHA-256: {documento['sha256']}")
print(f"Idioma: {documento['idioma']}")
print(f"Unidades no JSON: {len(documento['unidades'])}")
print(f"Título incluído nas contagens: {'sim (unidade 0)' if INCLUIR_TITULO_NA_CONTAGEM else 'não'}")
print(f"Unidades analisadas: {len(unidades)}; retiradas pelo recorte: {len(retiradas)}")
print("\nComposição do JSON (unidades por tipo e por categoria):")
exibir(pd.crosstab(
    pd.Series([u["tipo"] for u in documento["unidades"]], name="tipo"),
    pd.Series([u["categoria"] for u in documento["unidades"]], name="categoria"),
    margins=True, margins_name="Total",
), casas=0)

## 10. Pré-processamento e perfil do documento

*Visualização prévia: base inicial para a construção de novos gráficos (ver nota no início do notebook).*

A saída tem cinco partes: (a) filtros do recorte, com as unidades e os tokens que cada um retirou; (b) perfil do documento; (c) correções de lema aplicadas; (d) lemas de classes de conteúdo descartados pela lista de *stopwords*, com as suas frequências; (e) exame de inflação.

In [ ]:
nlp = spacy.load(MODELO_SPACY)
STOPWORDS = montar_stopwords(nlp)
tokens = processar_documento(unidades, nlp, STOPWORDS)
N_TOKENS = len(tokens)
N_UNIDADES = tokens["unidade"].nunique()

# (a) Filtros do recorte (skill 02, seção 3.1)
print("(a) Filtros do recorte: unidades e tokens retirados")
filtros = tabela_filtros(retiradas, nlp, STOPWORDS)
exibir(filtros, casas=0)

# (b) Perfil do documento (contagens sem casas decimais; médias e percentuais com duas)
print("(b) Perfil do documento")
perfil = perfil_documento(tokens, TIPO_UNIDADE, SIGLA).to_frame()
display(perfil.style.format(lambda v: num(v, 0) if float(v).is_integer() else num(v, 2)))
salvar_tabela(perfil, "00_perfil_documento")
salvar_tabela(tokens, "00_tokens_processados", index=False)
tokens_por_categoria = tokens["unidade"].map(CATEGORIA_UNIDADE).value_counts()
print("Tokens por categoria do JSON: " + "; ".join(
    f"{cat}: {num(n, 0)} ({num(n / N_TOKENS * 100, 1)}%)" for cat, n in tokens_por_categoria.items()))
print(f"Stopwords aplicadas: {len(STOPWORDS)} (lista do spaCy, {len(STOPWORDS_ADICIONAIS)} adicionada(s), "
      f"{len(STOPWORDS_MANTIDAS)} mantida(s))")

# (c) Correções explícitas de lema (CORRECOES_DE_LEMA)
correcoes = tabela_correcoes_lema(tokens)
salvar_tabela(correcoes, "00_correcoes_de_lema")
print(f"\n(c) Correções de lema aplicadas: {len(correcoes)} de {len(CORRECOES_DE_LEMA)} entradas da configuração "
      f"encontradas no texto, afetando {int(correcoes['ocorrencias'].sum())} tokens")
exibir(correcoes)

# (d) Lemas de classes de conteúdo que a lista de stopwords retirou da contagem
descartados = lemas_descartados_por_stopwords(tokens, STOPWORDS)
salvar_tabela(descartados, "00_lemas_descartados_stopwords")
print(f"\n(d) Lemas de classes de conteúdo descartados pela lista de stopwords: {len(descartados)} lemas, "
      f"{int(descartados['frequencia'].sum())} ocorrências "
      f"({num(descartados['frequencia'].sum() / N_TOKENS * 100, 1)}% dos tokens)")
exibir(descartados)

# (e) Exame de inflação (skill 02, seção 3.5): apenas diagnostica; nada é retirado aqui
identicas, repetidos = examinar_inflacao(unidades, MIN_PALAVRAS_TRECHO_REPETIDO)
alfanumericas = formas_alfanumericas(unidades, nlp)
print("\n(e) Exame de inflação (diagnóstico)")
print(f"Unidades com texto idêntico: {len(identicas)}")
if len(identicas):
    exibir(identicas)
print(f"Trechos de {MIN_PALAVRAS_TRECHO_REPETIDO} ou mais palavras repetidos em unidades diferentes: {len(repetidos)}")
if len(repetidos):
    exibir(repetidos)
print("Formas com letras e dígitos (não são tokens pela definição e ficam fora das contagens): "
      + ("; ".join(f"{f} ({n})" for f, n in alfanumericas.items()) if len(alfanumericas) else "nenhuma"))
print(f"Unidades retiradas como tratamento de inflação (UNIDADES_EXCLUIDAS): {len(UNIDADES_EXCLUIDAS)}")

**Método e escolhas.** Cada unidade é processada com o modelo `en_core_web_sm` do spaCy. Palavras compostas com hífen (*world-class*, *AI-optimised*) são reunidas em um só token. **Token** é toda palavra formada por letras (com hífen interno ou sigla com pontos); números e pontuação não são tokens. O total de tokens é o **denominador** de todas as taxas por 1.000 tokens. **Tokens de conteúdo** são substantivos, nomes próprios, verbos, adjetivos e advérbios fora da lista de *stopwords* do spaCy, verificada pelo **lema**, para que todas as flexões de uma palavra recebam o mesmo tratamento. Escolhas aplicadas a este plano:

- **Recorte (a):** o título entra como unidade 0; os títulos internos e as duas categorias do JSON (`Body` e `Box`) entram; nenhuma unidade foi retirada. Os quadros (`Box`) somam 968 tokens, 10,8% do total.
- **Correções de lema (c):** além de *datum* → *data* (59 ocorrências), o spaCy não lematiza nomes próprios no plural, o que separaria *AI Factories* (lema *factories*) de *AI Factory* (lema *factory*). Aplicou-se uma regra única: todo plural regular de nome próprio encontrado no texto recebe o lema singular, como o spaCy já faz com os substantivos comuns. São 27 lemas e 139 ocorrências, como *factories* (33), *states* (28), *hubs* (14), *gigafactories* (11) e *skills* (11). A lista fica em `CORRECOES_DE_LEMA` e pode ser revertida apagando as entradas.
- ***Stopwords* (d):** a lista do spaCy foi aplicada sem alterações (`STOPWORDS_ADICIONAIS` e `STOPWORDS_MANTIDAS` vazias, padrão da skill). A tabela mostra os 56 lemas de classes de conteúdo que ela retirou da contagem (298 ocorrências, 3,3% dos tokens).
- **Exame de inflação (e):** não há unidades idênticas nem resíduos de extração; as formas com dígitos são trimestres e siglas do próprio texto, e não marcadores de nota colados a palavras. Há um trecho de 31 palavras repetido nas unidades 109 e 110, as duas primeiras da seção 4: *the AI Continent will focus on measures to enlarge the EU’s pool of AI specialists and to adequately upskill and reskill EU workers and citizens in the use of AI*. Como o próprio plano escreveu a frase duas vezes, em parágrafos consecutivos, ela é **frequência real**, e não artefato de diagramação, e foi mantida (skill 02, seção 3.5). `UNIDADES_EXCLUIDAS` continua vazia.

A tabela de tokens processados (`00_tokens_processados.csv`, exportada para `resultados/` apenas por comando do pesquisador) permite conferir, token a token, a forma, o lema dado pelo spaCy, o lema final, a classe gramatical e a decisão de inclusão.

**Como ler.** A tabela (a) mostra o que o recorte retirou. O perfil (b) descreve a extensão do documento e do seu vocabulário e serve de referência para dimensionar as frequências apresentadas a seguir. As tabelas (c) e (d) mostram, respectivamente, o que as correções de lema reuniram e o que a lista de *stopwords* tirou da contagem de conteúdo. O exame (e) indica se há algo multiplicando termos artificialmente.

**Interpretação.** *Observação:* o plano tem 8.931 tokens em 147 unidades analisadas (o título, 18 títulos internos, 87 parágrafos e 41 itens de lista), nenhuma delas sem tokens. Os tokens de conteúdo são 5.276 (59,08% do total) e se distribuem em 1.253 lemas distintos, dos quais 579 (46,2%) aparecem uma única vez. O exame de inflação não encontrou artefatos que exigissem tratamento. *Inferência:* a coexistência de muitos lemas de ocorrência única com poucos lemas muito frequentes (seção 12) sugere um vocabulário amplo, organizado em torno de um núcleo pequeno de termos recorrentes. A leitura é provisória, porque a proporção de lemas de ocorrência única depende da extensão do texto.

**Fragilidades e ajustes possíveis.** O lematizador e o etiquetador do spaCy cometem erros, sobretudo em siglas e nomes próprios; por isso, as tabelas trazem as formas observadas de cada lema. Pontos específicos deste plano:

- A lista de *stopwords* do spaCy inclui *call*, que aqui tem uso substantivo (14 ocorrências, como em *call for expression of interest*, *Call for Proposals* e *Call for Evidence*). Mantê-la na contagem é decisão do pesquisador (`STOPWORDS_MANTIDAS = ["call"]`).
- A sigla *US* (1 ocorrência) recebe o lema *us*, que é *stopword*, e o substantivo *uses* (1) também recebe o lema *us*; as duas ocorrências ficam fora da contagem de conteúdo.
- Pela definição de token (só letras), as siglas *GenAI4EU* (4), *DIH4CAT* (1) e *3D* (2) e os trimestres (*Q1* a *Q4*) não entram em nenhuma contagem.
- A regra dos plurais de nome próprio segue o princípio da lematização, mas reúne sob o mesmo lema nomes de instrumentos e usos comuns da palavra (por exemplo, *Member States* e *State aid* ficam ambos em *state*).
- A frase repetida da seção 4 soma às contagens as ocorrências da segunda cópia (por exemplo, 3 de *ai* e 2 de *eu*). Retirá-la exigiria excluir uma unidade inteira, que tem outro conteúdo, e o efeito é pequeno (31 palavras, 0,3% dos tokens).

## 11. Extensão das unidades textuais

*Visualização prévia: base inicial para a construção de novos gráficos (ver nota no início do notebook).*

In [ ]:
tam_unidades = tokens.groupby("unidade").size().reindex([u for u, _ in unidades], fill_value=0)

fig, ax = plt.subplots(figsize=(7.5, 3.2))
x = np.arange(len(tam_unidades))
ax.bar(x, tam_unidades.to_numpy(), width=0.8, color=COR)
ax.axhline(tam_unidades.mean(), color=COR_TINTA_2, lw=0.8)
ax.text(len(x) - 0.5, tam_unidades.mean(), f"média = {num(tam_unidades.mean(), 1)}",
        ha="right", va="bottom", fontsize=9, color=COR_TINTA_2)
passo = max(1, int(np.ceil(len(x) / 20)))
ax.set_xticks(x[::passo], labels=[str(u) for u in tam_unidades.index[::passo]])
ax.set_xlim(-0.6, len(x) - 0.4)
ax.set_xlabel("Unidade (id; 0 = título), na ordem do documento")
ax.set_ylabel("Tokens (palavras)")
ax.set_title(f"{SIGLA} · extensão das unidades textuais", loc="left")
ax.grid(axis="x", visible=False)
ax.yaxis.set_major_formatter(FMT_BR)
salvar_figura(fig, "01_extensao_unidades")
plt.show()

salvar_tabela(tam_unidades.rename("tokens").to_frame(), "01_extensao_unidades")
maior, menor = tam_unidades.idxmax(), tam_unidades.idxmin()
print(f"Síntese factual: {len(tam_unidades)} unidades; média de {num(tam_unidades.mean(), 1)} tokens "
      f"(desvio-padrão {num(tam_unidades.std(), 1)}). Maior unidade: id {maior} ({tam_unidades[maior]} tokens); "
      f"menor: id {menor} ({tam_unidades[menor]} token{'' if tam_unidades[menor] == 1 else 's'}).")
cinco_maiores = tam_unidades.sort_values(ascending=False, kind="stable").head(5)
print("As cinco maiores unidades: " + "; ".join(f"id {u} ({n} tokens)" for u, n in cinco_maiores.items()) + ".")
por_tipo = tam_unidades.groupby(pd.Series(TIPO_UNIDADE)).agg(["count", "sum", "mean", "median", "min", "max"])
print("Por tipo de unidade: " + "; ".join(
    f"{ROTULO_TIPO[tipo]}: {int(l['count'])} unid., {num(l['sum'], 0)} tokens ({num(l['sum'] / N_TOKENS * 100, 1)}% do total), "
    f"média {num(l['mean'], 1)}, mediana {num(l['median'], 1)}, mín. {int(l['min'])}, máx. {int(l['max'])}"
    for tipo, l in por_tipo.reindex([t for t in ROTULO_TIPO if t in por_tipo.index]).iterrows()
) + ".")

**Método e escolhas.** Número de tokens (palavras) de cada unidade, na ordem original do documento, incluindo o título (unidade 0) e os títulos internos, que entram na contagem por padrão. A linha horizontal marca a média. Nenhuma unidade é omitida. A síntese abaixo do gráfico detalha a extensão por tipo de unidade.

**Como ler.** Cada barra é uma unidade: o título, um título interno, um parágrafo ou um item de lista. Barras altas indicam parágrafos extensos; sequências de barras baixas indicam títulos internos ou listas de itens curtos.

**Interpretação.** *Observação:* as 147 unidades têm, em média, 60,8 tokens (desvio-padrão 53,8), mas a média reúne tipos de unidade muito diferentes. Os 87 parágrafos concentram 8.027 tokens (89,9% do total), com mediana de 91 tokens; os 41 itens de lista somam 740 tokens (8,3%), com mediana de 16; os 18 títulos internos somam 160 tokens (1,8%). As maiores unidades são as de id 138 (246 tokens), 34 (242), 137 (192), 98 (187) e 76 (186); a menor é o título interno *Conclusion* (id 143, 1 token). O gráfico alterna blocos de parágrafos longos com sequências de barras baixas, como as dos ids 100 a 107 e 124 a 134, que são itens de lista; 36 dos 41 itens de lista pertencem aos quadros (`Box`, tabela da seção 9). *Inferência:* a argumentação do plano está quase toda nos parágrafos. Os itens de lista, em sua maioria nos quadros, são numerosos como unidades, mas ocupam pouco espaço em tokens.

**Fragilidades e ajustes possíveis.** A segmentação em unidades é a da skill 01, que segue a estrutura do original. Como os títulos internos e os itens de lista são curtos, a média por unidade (60,8) representa mal os parágrafos (mediana 91); a síntese por tipo corrige essa leitura. Unidades muito curtas tornam instáveis as medidas por unidade (alcance e dispersão), o que deve ser lembrado nas seções seguintes. Para analisar apenas o texto corrido, o pesquisador pode retirar os títulos internos (`INCLUIR_TITULOS_INTERNOS = False`) ou os quadros (`CATEGORIAS_EXCLUIDAS = ["Box"]`).

## 12. Lemas de conteúdo mais frequentes

*Visualização prévia: base inicial para a construção de novos gráficos (ver nota no início do notebook).*

In [ ]:
freq = tabela_frequencias(tokens)
salvar_tabela(freq, "02_frequencias_lemas_completa")
relatar_empates(freq, "frequencia", TOP_N_LEMAS, "lema")
top = freq.head(TOP_N_LEMAS)

fig, ax = plt.subplots(figsize=(7.5, 0.27 * len(top) + 1.3))
y = np.arange(len(top))
ax.barh(y, top["frequencia"], height=0.7, color=COR)
ax.set_yticks(y, labels=top.index)
ax.invert_yaxis()
xmax = top["frequencia"].max()
for yi, (_, linha) in zip(y, top.iterrows()):
    ax.text(linha["frequencia"] + xmax * 0.015, yi,
            f"{num(linha['por_1000_tokens'])} por 1.000 · {int(linha['unidades_com_ocorrencia'])}/{N_UNIDADES} unid.",
            va="center", fontsize=9, color=COR_TINTA_2)
ax.set_xlim(0, xmax * 1.55)
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.xaxis.set_major_formatter(FMT_BR)
ax.set_xlabel("Frequência absoluta (ocorrências)")
ax.set_title(f"{SIGLA} · {len(top)} lemas de conteúdo mais frequentes", loc="left")
ax.grid(axis="y", visible=False)
salvar_figura(fig, "02_lemas_mais_frequentes")
plt.show()

exibir(top[["frequencia", "por_1000_tokens", "unidades_com_ocorrencia", "pct_unidades", "DP_norm", "formas_observadas"]])
n_conteudo = int(tokens["conteudo"].sum())
print(f"Síntese factual: os {len(top)} lemas acima somam {int(top['frequencia'].sum())} ocorrências, "
      f"{num(top['frequencia'].sum() / n_conteudo * 100, 1)}% dos {n_conteudo} tokens de conteúdo. "
      f"O documento tem {len(freq)} lemas de conteúdo distintos (tabela completa exportada).")

**Método e escolhas.** Frequência absoluta dos *N* lemas de conteúdo mais frequentes (*N* = `TOP_N_LEMAS` = 25). O rótulo de cada barra traz a frequência por 1.000 tokens e o **alcance** (em quantas das 147 unidades o lema aparece). A tabela acrescenta o **DP normalizado** (dispersão) e as formas observadas. Ordenação: frequência decrescente; empates resolvidos por ordem alfabética e informados acima do gráfico quando afetam o corte (aqui não houve empate no 25º lugar). Os lemas já refletem as correções de `CORRECOES_DE_LEMA` (seção 10). A tabela completa de todos os lemas (`02_frequencias_lemas_completa.csv`) é exportada para `resultados/` apenas por comando do pesquisador.

**Como ler.** Barras mais longas indicam lemas mais frequentes. Um lema frequente presente em muitas unidades é estruturante no texto; um lema frequente presente em poucas unidades é localizado. A distinção aparece no rótulo (alcance) e na coluna DP_norm da tabela (0 = espalhado; 1 = concentrado).

**Interpretação.** *Observação:* *ai* domina o vocabulário de conteúdo, com 358 ocorrências (40,09 por 1.000 tokens), presentes em 127 das 147 unidades e com a menor dispersão do recorte (DP = 0,25). Seguem *eu* (107 ocorrências; 11,98 por 1.000; 62 unidades) e *data* (90; 10,08 por 1.000), este concentrado em 33 unidades (DP = 0,72). Os 25 lemas somam 1.363 ocorrências, 25,8% dos 5.276 tokens de conteúdo. Parte do topo vem de nomes próprios de instrumentos e instituições, como mostram os bigramas da seção 14: das 41 ocorrências de *factory*, 39 estão em *ai factory*; das 41 de *act*, 27 estão em *ai act*; as 26 de *member* estão todas em *member state*. Entre os verbos, aparecem *include* (49, sobretudo na forma *including*, 35), *support* (43), *ensure* (35), *launch* (27) e *need* (25). *Inferência:* o léxico mais frequente combina a referência à própria União (*eu*, *european*, *europe*, *commission*), os nomes dos instrumentos que o plano cria ou mobiliza (*AI Factories*, *AI Act*) e verbos de ação programática (*support*, *ensure*, *launch*). Esse perfil é compatível com um plano de ação, mas a frequência, por si só, não mede a prioridade política de cada tema.

**Fragilidades e ajustes possíveis.** Frequência não equivale a importância política: um termo pode ser frequente por razões de estilo ou por repetição formular. Lemas genéricos, como *ai* e *eu*, tendem a ocupar o topo em planos sobre IA. Isso é um resultado, e não um erro, e por isso esses lemas não são removidos sem decisão do pesquisador. *AI* e *artificial intelligence* não são unificados, porque isso alteraria o texto; se o pesquisador desejar, a unificação pode ser feita por regra explícita na configuração. Neste plano, nomes próprios alimentam lemas comuns: *AI Act* alimenta *act*; *European Digital Innovation Hubs* alimenta *european*, *digital* e *innovation*; *Apply AI Strategy* contribui com 15 ocorrências do lema *apply* (bigrama *apply ai*, seção 14). A contagem está correta, mas o lema isolado mistura o nome do instrumento com o uso comum da palavra; os bigramas ajudam a separar os dois casos. *include* é frequente sobretudo como conector (*including*). Os quadros (`Box`, 10,8% dos tokens) reúnem sobretudo itens de lista com ações e prazos, parte dos quais retoma ações descritas no corpo e reforça esses mesmos nomes; o pesquisador pode retirá-los com `CATEGORIAS_EXCLUIDAS = ["Box"]`.

## 13. Distribuição dos lemas ao longo do documento

*Visualização prévia: base inicial para a construção de novos gráficos (ver nota no início do notebook).*

In [ ]:
lemas_disp = freq.head(TOP_N_DISPERSAO).index.tolist()
inicios = (tokens.groupby("unidade")["posicao"].min() / N_TOKENS).to_numpy()
posicoes = [tokens.loc[tokens["conteudo"] & (tokens["lema"] == lema), "posicao"].to_numpy() / N_TOKENS
            for lema in lemas_disp]
rotulos = [f"{lema}  (DP = {num(freq.loc[lema, 'DP_norm'])})" for lema in lemas_disp]

fig, ax = plt.subplots(figsize=(7.5, 0.33 * len(lemas_disp) + 1.3))
grafico_dispersao_posicional(ax, posicoes, rotulos, inicios, COR)
ax.set_title(f"{SIGLA} · posição das ocorrências dos {len(lemas_disp)} lemas mais frequentes", loc="left")
salvar_figura(fig, "03_dispersao_lemas")
plt.show()

exibir(freq.head(TOP_N_DISPERSAO)[["frequencia", "unidades_com_ocorrencia", "DP_norm"]].sort_values("DP_norm"))

dp_disp = freq.loc[lemas_disp, "DP_norm"]
print(f"Síntese factual: entre os {len(lemas_disp)} lemas, o DP normalizado vai de {num(dp_disp.min())} "
      f"({dp_disp.idxmin()}, o mais espalhado) a {num(dp_disp.max())} ({dp_disp.idxmax()}, o mais concentrado).")
# Referência de leitura: com mais de 120 unidades, a função não desenha o início de cada unidade.
# Os títulos internos do plano (unidades analisadas, type = "heading") servem de marco.
inicio_unidade = tokens.groupby("unidade")["posicao"].min() / N_TOKENS
print("Referência de leitura: posição relativa em que começa cada título interno do plano:")
for uid, texto in unidades:
    if TIPO_UNIDADE[uid] == "heading":
        print(f"  {num(inicio_unidade[uid])}  (id {uid})  {texto}")

**Método e escolhas.** Cada traço vertical é uma ocorrência do lema, posicionada pela sua posição relativa no documento (posição do token ÷ total de tokens). O DP normalizado (Gries, 2008; Lijffijt & Gries, 2012) resume a distribuição: 0 indica ocorrências proporcionais ao tamanho das unidades; 1, concentração máxima. Como o plano tem 147 unidades (mais de 120), a função não desenha o início de cada unidade, para não encobrir os traços. Em seu lugar, a síntese abaixo do gráfico informa a posição em que começa cada título interno, como referência de leitura.

**Como ler.** Linhas com traços espalhados indicam termos presentes em todo o texto; traços agrupados indicam termos concentrados em uma parte (por exemplo, em uma seção). Para saber a que parte do plano corresponde um agrupamento, compare a posição dos traços com a lista de títulos internos impressa abaixo do gráfico. A tabela ordena os lemas do mais espalhado ao mais concentrado.

**Interpretação.** *Observação:* o DP normalizado vai de 0,25 (*ai*) a 0,82 (*factory*). *ai* aparece ao longo de todo o documento. *eu* (0,46), *include* e *european* (0,57) e *sector* (0,58) também se espalham, embora com intervalos. Três lemas formam agrupamentos nítidos, que coincidem com seções do plano: *factory* (0,82) se adensa entre 0,14 e 0,21, trecho que vai do título *Deploy and scale AI Factories* (id 16) ao título *Invest in AI Gigafactories* (id 30); *data* (0,72) se adensa entre 0,40 e 0,50, do título *Data for AI* (id 58) ao título *Foster innovation and accelerate AI adoption in strategic EU sectors* (id 72); *act* (0,77) se concentra a partir de 0,86, depois do título *Fostering regulatory compliance and simplification* (id 135). *commission* (0,68) aparece em todo o texto, mas fica mais denso a partir da seção 4 (título *Strengthen AI skills and talent*, id 108, posição 0,72). *Inferência:* parte do vocabulário mais frequente é transversal ao plano (*ai*, *eu*, *european*, *sector*), e parte é temática, restrita à seção que trata do assunto. No segundo caso, a frequência total depende muito da extensão da seção dedicada ao tema.

**Fragilidades e ajustes possíveis.** O DP é sensível à segmentação: uma unidade longa pode concentrar ocorrências por ser longa, e não por ênfase. O gráfico mostra posições, não intensidade. Sem as linhas de início das unidades, a correspondência com as seções depende da lista de títulos internos impressa abaixo do gráfico; nela, os quatro títulos de ids 86 a 92 (posições 0,60 a 0,63) são títulos dos casos apresentados em quadro (`Box`), e não seções do plano. Marcar as seções no próprio gráfico ou medir a dispersão por seção é possível com o campo `path`, por comando do pesquisador.

## 14. Expressões recorrentes (bigramas)

*Visualização prévia: base inicial para a construção de novos gráficos (ver nota no início do notebook).*

In [ ]:
bigramas = tabela_bigramas(tokens)
salvar_tabela(bigramas, "04_bigramas_completa")
elegiveis = bigramas[bigramas["frequencia"] >= FREQ_MINIMA_BIGRAMA]
relatar_empates(elegiveis, "frequencia", TOP_N_BIGRAMAS, "bigrama")
topb = elegiveis.head(TOP_N_BIGRAMAS)

if topb.empty:
    print(f"Nenhum bigrama atinge a frequência mínima de {FREQ_MINIMA_BIGRAMA}; gráfico não gerado.")
else:
    fig, ax = plt.subplots(figsize=(7.5, 0.27 * len(topb) + 1.3))
    y = np.arange(len(topb))
    ax.barh(y, topb["frequencia"], height=0.7, color=COR)
    ax.set_yticks(y, labels=topb["forma_principal"])
    ax.invert_yaxis()
    xmax = topb["frequencia"].max()
    for yi, (_, linha) in zip(y, topb.iterrows()):
        ax.text(linha["frequencia"] + xmax * 0.015, yi,
                f"{int(linha['unidades_com_ocorrencia'])}/{N_UNIDADES} unid.",
                va="center", fontsize=9, color=COR_TINTA_2)
    ax.set_xlim(0, xmax * 1.25)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.xaxis.set_major_formatter(FMT_BR)
    ax.set_xlabel("Frequência absoluta (ocorrências)")
    ax.set_title(f"{SIGLA} · {len(topb)} expressões de duas palavras mais frequentes", loc="left")
    ax.grid(axis="y", visible=False)
    salvar_figura(fig, "04_bigramas")
    plt.show()
    exibir(topb)
print(f"Síntese factual: {len(bigramas)} bigramas distintos; {len(elegiveis)} com frequência >= {FREQ_MINIMA_BIGRAMA}.")

**Método e escolhas.** Pares de tokens de conteúdo **adjacentes no texto original**, na mesma frase e sem pontuação entre eles. Os pares são agrupados pelo lema (ex.: *member state* reúne *member states*), e o rótulo mostra a forma escrita mais frequente. Só entram no gráfico os bigramas com frequência mínima `FREQ_MINIMA_BIGRAMA` (2). Como a adjacência é verificada no texto original, a remoção de *stopwords* não cria pares artificiais (em *pool of AI specialists*, não se forma *pool ai*). O empate no ponto de corte dos 20 bigramas está informado acima do gráfico.

**Como ler.** As barras mostram as expressões que o texto mais repete, frequentemente nomes de instrumentos e de conceitos. O rótulo indica em quantas unidades a expressão aparece: uma expressão frequente em poucas unidades está concentrada em uma parte do texto.

**Interpretação.** *Observação:* há 1.488 bigramas distintos, dos quais 259 aparecem duas vezes ou mais. O mais frequente é *ai factories* (39 ocorrências, em 23 unidades), seguido de *ai act* (27, em apenas 9 unidades) e de *member states* (26, em 20 unidades). Boa parte dos 20 primeiros são nomes de instrumentos e iniciativas ou fragmentos deles: *apply ai* e *ai strategy* (de *Apply AI Strategy*); *european digital*, *digital innovation* e *innovation hubs* (de *European Digital Innovation Hubs*); *ai gigafactories*; *data labs*. Ao lado deles aparecem expressões de política pública (*public sector*, 15; *public consultation*, 10; *single market*, 9) e de infraestrutura e tecnologia (*ai models*, 19; *data centre*, 13; *data spaces*, 12). *Inferência:* as expressões recorrentes são, em grande parte, os nomes dos instrumentos que o plano cria ou mobiliza. Isso indica um vocabulário organizado em torno de iniciativas institucionais nomeadas, mais do que de conceitos abstratos.

**Fragilidades e ajustes possíveis.** Expressões com três ou mais palavras aparecem fragmentadas em bigramas sobrepostos: *European Digital Innovation Hubs* gera três bigramas (*european digital*, *digital innovation*, *innovation hubs*) e *Apply AI Strategy* gera dois (*apply ai*, *ai strategy*), o que multiplica a presença do mesmo nome no recorte. Expressões interrompidas por *stopwords* (*pool of AI specialists*) não são captadas como bigramas. Os bigramas *european ai* e *public administration* empatam com *data labs* (8 ocorrências) e ficaram fora do recorte pela ordem alfabética. Uma análise de n-gramas mais longos ou dos nomes de instrumentos pode ser feita por comando do pesquisador.

## 15. Lemas de conteúdo mais frequentes: recorte de 30 lemas

*Visualização construída por comando do pesquisador: expandir a análise de unigramas de 25 para 30 lemas, sem retirar nenhum termo. As visualizações anteriores (seções 10 a 14) foram preservadas sem alteração.*

In [ ]:
# Parâmetro desta visualização (comando do pesquisador: análise de 30 unigramas).
# Fica nesta célula, e não na célula de configuração, para que as células anteriores
# permaneçam idênticas. Os dados são os da seção 12: a tabela freq, com todos os lemas.
TOP_N_LEMAS_COMANDO = 30

relatar_empates(freq, "frequencia", TOP_N_LEMAS_COMANDO, "lema")
top30 = freq.head(TOP_N_LEMAS_COMANDO)

fig, ax = plt.subplots(figsize=(7.5, 0.27 * len(top30) + 1.3))
y = np.arange(len(top30))
ax.barh(y, top30["frequencia"], height=0.7, color=COR)
ax.set_yticks(y, labels=top30.index)
ax.invert_yaxis()
xmax = top30["frequencia"].max()
for yi, (_, linha) in zip(y, top30.iterrows()):
    ax.text(linha["frequencia"] + xmax * 0.015, yi,
            f"{num(linha['por_1000_tokens'])} por 1.000 · {int(linha['unidades_com_ocorrencia'])}/{N_UNIDADES} unid.",
            va="center", fontsize=9, color=COR_TINTA_2)
ax.set_xlim(0, xmax * 1.55)
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.xaxis.set_major_formatter(FMT_BR)
ax.set_xlabel("Frequência absoluta (ocorrências)")
ax.set_title(f"{SIGLA} · {len(top30)} lemas de conteúdo mais frequentes", loc="left")
ax.grid(axis="y", visible=False)
salvar_figura(fig, "05_lemas_mais_frequentes_30")
plt.show()

exibir(top30[["frequencia", "por_1000_tokens", "unidades_com_ocorrencia", "pct_unidades", "DP_norm", "formas_observadas"]])
n_conteudo = int(tokens["conteudo"].sum())
acrescidos = top30.index[TOP_N_LEMAS:]
print(f"Síntese factual: os {len(top30)} lemas acima somam {int(top30['frequencia'].sum())} ocorrências, "
      f"{num(top30['frequencia'].sum() / n_conteudo * 100, 1)}% dos {n_conteudo} tokens de conteúdo. "
      f"Em relação ao recorte de {TOP_N_LEMAS} lemas (seção 12), entram: "
      + "; ".join(f"{lema} ({int(top30.loc[lema, 'frequencia'])})" for lema in acrescidos) + ".")

**Método e escolhas.** Mesma medida e mesmo gráfico da seção 12, com recorte de 30 lemas em vez de 25, por comando do pesquisador. Nenhum termo foi retirado: os dados são a mesma tabela `freq`, com todos os lemas de conteúdo, os mesmos parâmetros de pré-processamento e as mesmas correções de lema da seção 10. O parâmetro `TOP_N_LEMAS_COMANDO = 30` fica no início da própria célula, e não na célula de configuração, para que as células anteriores permaneçam idênticas. Ordenação: frequência decrescente; empates resolvidos por ordem alfabética. Houve empate no 30º lugar, informado acima do gráfico.

**Como ler.** Igual à seção 12: barras mais longas indicam lemas mais frequentes; o rótulo traz a frequência por 1.000 tokens e o alcance (em quantas das 147 unidades o lema aparece); a coluna DP_norm da tabela indica a dispersão (0 = espalhado; 1 = concentrado). Os 25 primeiros lemas são os mesmos da seção 12, na mesma ordem.

**Interpretação.** *Observação:* os 30 lemas somam 1.475 ocorrências, 28,0% dos 5.276 tokens de conteúdo (os 25 da seção 12 somavam 25,8%). Os cinco lemas acrescentados têm frequências próximas: *apply*, *initiative* e *skill* (23 cada), *solution* (22) e *action* (21). Todos são relativamente concentrados (DP_norm de 0,76 a 0,87); *skill* é o mais concentrado (0,87) e o de menor alcance (14 unidades). *Inferência:* a ampliação acrescenta termos ligados à execução do plano (*initiative*, *action*, *solution*) e termos ligados a duas das suas frentes, a adoção da IA (*apply*, em boa parte pelo nome *Apply AI Strategy*) e as competências (*skill*).

**Fragilidades e ajustes possíveis.** Valem as fragilidades da seção 12: frequência não equivale a importância política, e nomes próprios alimentam lemas comuns. Pontos específicos do recorte de 30:

- **Empate no corte:** *action* (21) entra no recorte, e *application*, *cloud* e *facilitate*, também com 21, ficam de fora apenas pela ordem alfabética. Entre esses quatro lemas, a fronteira dos 30 é arbitrária.
- ***skill*** chega a 23 ocorrências por causa da correção dos plurais de nome próprio (seção 10, tabela c): 11 delas vêm de *Skills* em nomes como *AI Skills Academy* e *Union of Skills*. Sem a correção, *skill* teria 12 ocorrências e ficaria fora do recorte.
- ***apply*** inclui o nome *Apply AI Strategy* (a forma *apply ai* aparece 15 vezes, seção 14), e *action* inclui nomes como *AI Continent Action Plan*, o título do plano, e *AI Action Summit*.